In [34]:
#!/usr/bin/env python3
# combine_crop_ocr.py  (v3: new crop & 50-page progress messages)

from pathlib import Path
import fitz, cv2, numpy as np, pytesseract, re
from collections import defaultdict

# ───── configuration ──────────────────────────────────────────────────
IN_DIR   = Path("../data/mini")
OUT_PDF  = Path("../case_outputs/a.pdf")
CROP_PX  = (40, 20, 540, 70)      # (x0, y0, x1, y1)  ← changed x0 from 50→40
CROP_DPI = 144                    # dpi for cropping pass
OCR_DPI  = 300                    # dpi for OCR pass
WORD     = "Benton"
RECURSE  = True                   # search sub-folders for PDFs?
# ──────────────────────────────────────────────────────────────────────


def concat_and_crop_bitmap(in_dir: Path, out_file: Path,
                           crop_px, crop_dpi=144, recurse=True):
    """
    Build cropped-only PDF.
    Returns: total pages written, and an index map (out_idx → (src_path, src_pg))
    """
    x0, y0, x1, y1 = crop_px
    w_px, h_px = x1 - x0, y1 - y0
    w_pt, h_pt = w_px * 72 / crop_dpi, h_px * 72 / crop_dpi
    clip_rect = fitz.Rect(x0 * 72 / crop_dpi,
                          y0 * 72 / crop_dpi,
                          x1 * 72 / crop_dpi,
                          y1 * 72 / crop_dpi)

    out_doc   = fitz.open()
    index_map = []
    finder    = in_dir.rglob if recurse else in_dir.glob
    pdf_files = sorted(finder("*.pdf")) + sorted(finder("*.PDF"))
    if not pdf_files:
        raise FileNotFoundError(f"No PDFs found in {in_dir}")

    page_count = 0
    for pdf in pdf_files:
        with fitz.open(pdf) as src:
            for pg in src:
                pix = pg.get_pixmap(dpi=crop_dpi,
                                    clip=clip_rect,
                                    colorspace=fitz.csRGB,
                                    alpha=False)
                dst = out_doc.new_page(width=w_pt, height=h_pt)
                dst.insert_image(fitz.Rect(0, 0, w_pt, h_pt),
                                 stream=pix.tobytes("png"))
                index_map.append((pdf, pg.number + 1))
                page_count += 1

    if not index_map:
        out_doc.close()
        raise ValueError("No pages processed; nothing to save")

    out_doc.save(out_file, garbage=4, deflate=True)
    out_doc.close()
    return page_count, index_map


def find_word_locations(out_pdf: Path, word: str, index_map, dpi=300):
    """
    OCR each cropped page; return dict{src_path → [src_pages]} for pages containing *word*.
    Shows progress every 50 pages.
    """
    regex = re.compile(rf"\b{re.escape(word)}\b", re.IGNORECASE)
    hits  = defaultdict(list)

    with fitz.open(out_pdf) as doc:
        for out_idx, page in enumerate(doc, 1):                # 1-based for progress
            pix = page.get_pixmap(dpi=dpi, colorspace=fitz.csRGB, alpha=False)
            img = np.frombuffer(pix.samples, np.uint8).reshape(
                    pix.height, pix.width, pix.n)              # H × W × 3 (RGB)
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            thr  = cv2.threshold(gray, 0, 255,
                                 cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
            if regex.search(pytesseract.image_to_string(thr)):
                src_path, src_pg = index_map[out_idx - 1]      # index_map is 0-based
                hits[src_path].append(src_pg)

            if out_idx % 50 == 0:
                print(f"[EVENT] Extraction progress: {out_idx}/{len(index_map)} pages scanned")

    return hits


if __name__ == "__main__":
    pages, idx_map = concat_and_crop_bitmap(IN_DIR, OUT_PDF,
                                            crop_px=CROP_PX,
                                            crop_dpi=CROP_DPI,
                                            recurse=RECURSE)
    print(f"✓ Cropped & concatenated {pages} pages → {OUT_PDF}")

    occurrences = find_word_locations(OUT_PDF, WORD, idx_map, dpi=OCR_DPI)
    if not occurrences:
        print(f'"{WORD}" not found in any source file.')
    else:
        print(f'\nPages containing "{WORD}":')
        for pdf_path in sorted(occurrences):
            page_list = ", ".join(map(str, sorted(occurrences[pdf_path])))
            print(f"  • {pdf_path}  →  page(s): {page_list}")


…cropping progress: 50 pages processed
…cropping progress: 100 pages processed
…cropping progress: 150 pages processed
…cropping progress: 200 pages processed
…cropping progress: 250 pages processed
…cropping progress: 300 pages processed
…cropping progress: 350 pages processed
…cropping progress: 400 pages processed
…cropping progress: 450 pages processed
…cropping progress: 500 pages processed
…cropping progress: 550 pages processed
…cropping progress: 600 pages processed
…cropping progress: 650 pages processed
…cropping progress: 700 pages processed
…cropping progress: 750 pages processed
…cropping progress: 800 pages processed
…cropping progress: 850 pages processed
…cropping progress: 900 pages processed
…cropping progress: 950 pages processed
✓ Cropped & concatenated 963 pages → ../case_outputs/a.pdf
…OCR progress: 50/963 pages scanned
…OCR progress: 100/963 pages scanned
…OCR progress: 150/963 pages scanned
…OCR progress: 200/963 pages scanned
…OCR progress: 250/963 pages scanne